In [ ]:
!pip install neuralforecast -q
!pip install kaleido==0.2.1 -q
!pip install workalendar -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 71.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.5 which is incompatible.
   ━━━━━━━

In [ ]:
import pandas as pd
import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, TFT
from neuralforecast.losses.pytorch import MAE
from workalendar.europe import Russia
from tqdm import tqdm
from neuralforecast.models import TFT

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

cal = Russia()
def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8, "8h": 16, "24h": 48,
    "7d":  336, "14d": 672, "1m": 1488,
}

print(f"Строк: {len(df)}")
print(f"Дома: {houses}")

Строк: 36260
Дома: ['house_1', 'house_2', 'house_3', 'house_4', 'house_5', 'house_6', 'house_7', 'house_8']


In [ ]:
# neuralforecast требует формат: unique_id | ds | y | exog_features
n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

dfs = []
for house in houses:
    tmp = pd.DataFrame({
        "unique_id": house,
        "ds": df["timestamp"],
        "y": df[house],
        "temp_c": df["temp_c"],
        "humidity": df["humidity"],
        "cloudiness": df["cloudiness"],
        "hour": df["timestamp"].dt.hour,
        "weekday": df["timestamp"].dt.weekday,
        "month": df["timestamp"].dt.month,
        "is_weekend": (df["timestamp"].dt.weekday >= 5).astype(int),
        "is_holiday": df["is_holiday"],
    })
    dfs.append(tmp)

df_nf = pd.concat(dfs, ignore_index=True)

# разбивка
df_train = df_nf[df_nf["ds"] < df["timestamp"].iloc[train_end]]
df_val = df_nf[
    (df_nf["ds"] >= df["timestamp"].iloc[train_end]) &
    (df_nf["ds"] < df["timestamp"].iloc[val_end])
]
df_test = df_nf[df_nf["ds"] >= df["timestamp"].iloc[val_end]]
df_test

,unique_id,ds,y,temp_c,humidity,cloudiness,hour,weekday,month,is_weekend,is_holiday
30821,house_1,2019-08-15 03:00:00,31.23,17.000000,81.000000,100.000000,3,3,8,0,0
30822,house_1,2019-08-15 03:30:00,29.61,17.050000,81.833333,99.166667,3,3,8,0,0
30823,house_1,2019-08-15 04:00:00,30.60,17.100000,82.666667,98.333333,4,3,8,0,0
30824,house_1,2019-08-15 04:30:00,28.35,17.150000,83.500000,97.500000,4,3,8,0,0
30825,house_1,2019-08-15 05:00:00,29.37,17.200000,84.333333,96.666667,5,3,8,0,0
...,...,...,...,...,...,...,...,...,...,...,...
290075,house_8,2019-12-06 08:00:00,28.50,2.066667,89.000000,100.000000,8,4,12,0,0
290076,house_8,2019-12-06 08:30:00,28.50,2.083333,89.000000,100.000000,8,4,12,0,0
290077,house_8,2019-12-06 09:00:00,28.50,2.100000,89.000000,100.000000,9,4,12,0,0
290078,house_8,2019-12-06 09:30:00,28.50,2.133333,87.166667,100.000000,9,4,12,0,0


In [ ]:
nbeats_results = {}

for horizon_name, horizon_points in tqdm(horizons.items(), desc="NBEATSx горизонты"):
    print(f"\nГоризонт {horizon_name}")

    nf = NeuralForecast(
        models=[
            NBEATSx(
                h=horizon_points,
                input_size=336,
                max_steps=500,
                val_check_steps=50,
                early_stop_patience_steps=5,
                scaler_type="minmax",
                accelerator="gpu",
                devices=1,
            )
        ],
        freq="30min"
    )

    # используем только unique_id, ds, y
    df_nf_simple = df_nf[["unique_id", "ds", "y"]].copy()

    cv_df = nf.cross_validation(
        df=df_nf_simple,
        val_size=len(df_val) // len(houses),
        test_size=len(df_test) // len(houses),
        n_windows=None,
    )

    nbeats_results[horizon_name] = {
        house: evaluate(
            cv_df[cv_df["unique_id"] == house]["y"].values,
            cv_df[cv_df["unique_id"] == house]["NBEATSx"].values
        )
        for house in houses
    }

# сводная таблица
rows = []
for horizon_name in horizons.keys():
    for house in houses:
        metrics = nbeats_results[horizon_name][house]
        rows.append({
            "модель": "NBEATSx",
            "дом": house,
            "горизонт": horizon_name,
            "MAE": metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

df_nbeats = pd.DataFrame(rows)

for metric in ["MAE", "RMSE", "MAPE"]:
    pivot = df_nbeats.pivot(index="горизонт", columns="дом", values=metric)
    pivot = pivot.reindex(list(horizons.keys()))
    print(f"\nNBEATSx - {metric}:")
    print(pivot.to_string())

df_nbeats.to_csv("results_nbeats.csv", index=False)

Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1 0:00:00 • 0:00:00 0.00it/s

NBEATSx горизонты: 100%|██████████| 6/6 [04:16<00:00, 42.67s/it]


NBEATSx - MAE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          2.451    1.428    1.085    1.762    1.493    3.419    3.181    1.630
8h          2.640    1.496    1.125    1.863    1.572    3.650    3.437    1.729
24h         2.875    1.583    1.177    1.996    1.648    3.970    3.787    1.871
7d          3.947    1.929    1.386    2.570    2.076    5.169    4.927    2.498
14d         4.737    2.208    1.548    3.039    2.479    6.190    5.867    2.934
1m          6.097    2.670    1.813    3.793    3.107    7.937    7.395    3.740

NBEATSx - RMSE:
дом       house_1  house_2  house_3  house_4  house_5  house_6  house_7  house_8
горизонт                                                                        
4h          3.339    1.923    1.445    2.375    2.030    4.550    4.271    2.162
8h          3.635    2.018    1.504    2.523    2.143    4.881    4.638    2

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

2448

In [ ]:
df_nf_simple = df_nf[["unique_id", "ds", "y"]].copy()
val_size = len(df_val) // len(houses)
test_size = len(df_test) // len(houses)

In [ ]:
# переобучаем NBEATSx для дома 1
torch.cuda.empty_cache()
gc.collect()

horizon_name = "24h"
horizon_points = horizons[horizon_name]

nf_plot = NeuralForecast(
    models=[
        NBEATSx(
            h=horizon_points,
            input_size=336,
            max_steps=500,
            val_check_steps=50,
            early_stop_patience_steps=5,
            scaler_type="minmax",
            accelerator="gpu",
            devices=1,
        )
    ],
    freq="30min"
)

cv_df_plot = nf_plot.cross_validation(
    df=df_nf_simple,
    val_size=val_size,
    test_size=test_size,
    n_windows=None,
)

# берём только дом 1
house = "house_1"
house_cv = cv_df_plot[cv_df_plot["unique_id"] == house].reset_index(drop=True)

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

cutoffs = house_cv["cutoff"].unique()

for i, (hn, hp) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    # для коротких горизонтов берём один cutoff
    # для длинных - склеиваем несколько непересекающихся окон
    n_windows = max(1, hp // 48)
    selected_cutoffs = cutoffs[::48][:n_windows]

    subsets = []
    for c in selected_cutoffs:
        tmp = house_cv[house_cv["cutoff"] == c]
        subsets.append(tmp)

    subset = pd.concat(subsets).drop_duplicates(subset="ds").sort_values("ds")

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=subset["y"],
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=subset["ds"], y=subset["NBEATSx"],
        mode="lines", name="прогноз NBEATSx",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="NBEATSx: Прогнозные и фактические значения по всем горизонтам (дом 1)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("34_nbeats_forecast_all_horizons.png", scale=2)

INFO:lightning_fabric.utilities.seed:Seed set to 1
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name         ┃ Type          ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ loss         │ MAE           │      0 │ train │     0 │
│ 1 │ padder_train │ ConstantPad1d │      0 │ train │     0 │
│ 2 │ scaler       │ TemporalNorm  │      0 │ train │     0 │
│ 3 │ blocks       │ ModuleList    │  3.2 M │ train │     0 │
└───┴──────────────┴───────────────┴────────┴───────┴───────┘

Trainable params: 3.2 M                                                                                            
Non-trainable params: 37.2 K                                                                                       
Total params: 3.2 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 31                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: FutureWarning:

`isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` 
instead.

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=500` reached.


INFO:pytorch_lightning.utilities.rank_zero:Trainer already configured with model summary callbacks: [<class 'pytorch_lightning.callbacks.rich_model_summary.RichModelSummary'>]. Skipping setting a default `ModelSummary` callback.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Output()